# Step 3b – Cascade ASR → MT (নতুন ASR মডেল + আগের NLLB)
**Runtime → Change runtime type → T4 GPU (বা L4)**, তারপর **Run all**।

- ASR আবার চালাতে হয় না: Step 2d / 2e test-এ যে লেখা বানিয়েছে (`predictions_*.csv`), সেটাই NLLB দিয়ে বাংলা আর ইংরেজিতে অনুবাদ হয়।
- MT মডেল বদলায় না (Step 3-এর fine-tuned NLLB)।
- প্রথমবার সেকশন ১ library বদলে **নিজে থেকেই session restart** করতে পারে ("session crashed" দেখাবে) — তখন আবার **Run all** দিন।
- ফল সেভ হয় `Sylheti_Thesis/sylheti_mt/cascade_4spk`-এ। মাঝপথে কেটে গেলে Run all দিলে আগের অনুবাদ আবার করে না।

## 1. Drive আর library

In [ ]:
import os, sys, subprocess
from importlib.metadata import version, PackageNotFoundError
NEED = {"transformers": "4.48.3", "sacrebleu": "2.4.3"}
def _ver(p):
    try:
        return version(p)
    except PackageNotFoundError:
        return None
wrong = [p for p, v in NEED.items() if _ver(p) != v]
if wrong:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "sentencepiece"] +
                   [p + "==" + v for p, v in NEED.items()], check=True)
    print("installed", wrong, "-> restarting the session; press Run all again")
    os.kill(os.getpid(), 9)

from google.colab import drive
if not os.path.ismount("/content/drive"):
    drive.mount("/content/drive")
import torch
assert torch.cuda.is_available(), "GPU নেই: Runtime -> Change runtime type -> T4 GPU"
print(torch.cuda.get_device_name(0), "| transformers", version("transformers"))

## 2. কোন ASR মডেলের লেখা অনুবাদ হবে

In [ ]:
import re, unicodedata, pandas as pd
ROOT = "/content/drive/MyDrive/Sylheti_Thesis/Ultimate Thisis Data"
ASR_DIR = "/content/drive/MyDrive/Sylheti_Thesis/sylheti_asr"
MT_DIR = "/content/drive/MyDrive/Sylheti_Thesis/sylheti_mt"
MT_MODEL = MT_DIR + "/nllb-sylheti/final"
OUT = MT_DIR + "/cascade_4spk"; os.makedirs(OUT, exist_ok=True)

# ASR system -> folder with predictions_<tag>_<test set>.csv (missing ones are skipped)
SYSTEMS = {
    "Whisper-large-v3-turbo (fine-tuned)": (ASR_DIR + "/whisper-large-v3-turbo-4spk", "ft"),
    "Whisper-small (fine-tuned)":          (ASR_DIR + "/whisper-small-4spk", "ft"),
}
TEST_SETS = ["test_seen", "test_unseen"]
LANGS = {"bn": ("bangla", "ben_Beng"), "en": ("english", "eng_Latn")}

def clean(s):                       # same cleaning as Step 3 (MT was trained on this style)
    s = unicodedata.normalize("NFC", str(s))
    s = re.sub(r"[\"\'“”‘’(),.।?!:;\-/*…\[\]|]", " ", s)
    return re.sub(r"\s+", " ", s).strip()

corpus = pd.read_csv(ROOT + "/corpus.csv", encoding="utf-8-sig")
ref = corpus[["excel_row", "sylheti_asr", "bangla", "english", "split"]]

found = {}
for name, (folder, tag) in SYSTEMS.items():
    for ts in TEST_SETS:
        f = f"{folder}/predictions_{tag}_{ts}.csv"
        if os.path.exists(f):
            found[(name, ts)] = f
        else:
            print("not found (skipped):", f)
print("will translate:")
for (n, ts), f in found.items():
    print(" ", n, "|", ts, "|", os.path.basename(f))

# safety check: the MT model must not have seen these sentences in training
for (n, ts), f in found.items():
    rows = pd.read_csv(f).excel_row
    sp = ref.set_index("excel_row").loc[rows.unique(), "split"].value_counts().to_dict()
    print(f"{n} | {ts}: MT split of these sentences -> {sp}")

## 3. MT মডেল লোড

In [ ]:
import gc, sacrebleu
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
tok = AutoTokenizer.from_pretrained(MT_MODEL, src_lang="ben_Beng")
mt = AutoModelForSeq2SeqLM.from_pretrained(MT_MODEL, torch_dtype=torch.float16).to("cuda").eval()

def translate(texts, lang, bs=32, beams=4):
    out, i = [], 0
    bos = tok.convert_tokens_to_ids(lang)
    while i < len(texts):
        batch = [t if t else "-" for t in texts[i:i + bs]]
        enc = tok(batch, return_tensors="pt", padding=True, truncation=True, max_length=256).to("cuda")
        n_new = min(256, enc["input_ids"].shape[1] * 2 + 10)
        try:
            with torch.no_grad():
                ids = mt.generate(**enc, forced_bos_token_id=bos, num_beams=beams, max_new_tokens=n_new)
        except torch.cuda.OutOfMemoryError:
            del enc; gc.collect(); torch.cuda.empty_cache()
            if bs == 1:
                raise
            bs = max(1, bs // 2); print("out of memory -> batch size", bs); continue
        out += tok.batch_decode(ids, skip_special_tokens=True)
        i += len(batch)
    return out

def score(hyp, refs):
    return (round(sacrebleu.corpus_bleu(list(hyp), [list(refs)]).score, 2),
            round(sacrebleu.corpus_chrf(list(hyp), [list(refs)]).score, 2))
print("MT model ready:", MT_MODEL)

## 4. অনুবাদ আর BLEU / chrF
`gold Sylheti text` সারি = একই রেকর্ডিংগুলোর সঠিক সিলেটি লেখা থেকে অনুবাদ (ASR ভুল না করলে যা পাওয়া যেত — ঊর্ধ্বসীমা)।

In [ ]:
from tqdm.auto import tqdm
rows = []
for (name, ts), f in tqdm(list(found.items())):
    safe = re.sub(r"[^A-Za-z0-9]+", "_", name).strip("_")
    cache = f"{OUT}/predictions_{safe}_{ts}.csv"
    if os.path.exists(cache):
        a = pd.read_csv(cache, encoding="utf-8-sig")
    else:
        a = pd.read_csv(f).merge(ref, on="excel_row", how="inner")
        a["asr"] = a.pred.fillna("").map(clean)
        a["gold"] = a.sylheti_asr.map(clean)
        for k, (col, code_) in LANGS.items():
            a["mt_" + k] = translate(a.asr.tolist(), code_)
        a.to_csv(cache, index=False, encoding="utf-8-sig")
    for k, (col, code_) in LANGS.items():
        b, ch = score(a["mt_" + k].fillna(""), a[col])
        rows.append({"input": name, "test set": ts, "direction": "syl->" + k, "files": len(a), "BLEU": b, "chrF": ch})

# upper bound: gold Sylheti text of the same recordings (translated once per sentence)
gold_cache = OUT + "/gold_translations.csv"
need = sorted({r for f in found.values() for r in pd.read_csv(f).excel_row})
g = ref[ref.excel_row.isin(need)].copy()
cached = pd.read_csv(gold_cache, encoding="utf-8-sig") if os.path.exists(gold_cache) else None
if cached is not None and set(need) <= set(cached.excel_row):
    g = cached
else:
    g["gold"] = g.sylheti_asr.map(clean)
    for k, (col, code_) in LANGS.items():
        g["mt_" + k] = translate(g.gold.tolist(), code_)
    g.to_csv(gold_cache, index=False, encoding="utf-8-sig")
for ts in TEST_SETS:
    fs = [f for (n, t), f in found.items() if t == ts]
    if not fs:
        continue
    recs = pd.read_csv(fs[0])[["excel_row"]].merge(g, on="excel_row", how="inner")
    for k, (col, code_) in LANGS.items():
        b, ch = score(recs["mt_" + k].fillna(""), recs[col])
        rows.append({"input": "gold Sylheti text", "test set": ts, "direction": "syl->" + k, "files": len(recs), "BLEU": b, "chrF": ch})

res = pd.DataFrame(rows)
res.to_csv(OUT + "/cascade_results.csv", index=False)
wide = res.pivot_table(index=["input", "test set"], columns="direction", values=["BLEU", "chrF"]).round(2)
wide.columns = [f"{d} {m}" for m, d in wide.columns]
print(wide[[c for c in ["syl->bn BLEU", "syl->bn chrF", "syl->en BLEU", "syl->en chrF"] if c in wide.columns]].to_string())
print("\nsaved:", OUT + "/cascade_results.csv")

## 5. কয়েকটা উদাহরণ (প্রথম ASR মডেল, test_unseen)

In [ ]:
key = next((k for k in found if k[1] == "test_unseen"), next(iter(found)))
safe = re.sub(r"[^A-Za-z0-9]+", "_", key[0]).strip("_")
d = pd.read_csv(f"{OUT}/predictions_{safe}_{key[1]}.csv", encoding="utf-8-sig").sample(5, random_state=1)
for _, r in d.iterrows():
    print("সিলেটি (আসল):", r.gold)
    print("ASR লিখেছে  :", r.asr)
    print("বাংলা (আসল) :", r.bangla); print("বাংলা (cascade):", r.mt_bn)
    print("English (ref):", r.english); print("English (cascade):", r.mt_en); print()